In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scite.beam.continuous import ContinuousBeamModes
from scite.beam.continuous.continuous_beam_var_cs import ContinuousBeamVarCS, ContinuousBeamVarCSModes

## Reference cross-section

Same geometry as notebook `01_uls_state.ipynb`:
rectangular 300 × 700 mm, C30/37, B500B.

- **Uniform model** (`cba_uni`): doubly-reinforced section over the full span
  — compression (bottom) bars at the hogging support reduce x/d below the EC2
  singly-reinforced formula.
- **Variable-CS model** (`triple_var` / `cba_var`): segmented reinforcement matching actual
  detailing — top bars only in the hogging zone, bottom bars only in the
  sagging zone, both layers in the transition zone.  Available in all three
  material modes (design / characteristic / mean) via `ContinuousBeamVarCSModes`.

In [ ]:
b_ref, h_ref = 300.0, 700.0
A_s_bot_ref  = 1257.0      # 4×Ø20
A_s_top_ref  = 1610.0      # 4×Ø22
z_s_bot      = 50.0
z_s_top      = h_ref - z_s_bot   # 650.0 mm from bottom
L_ref        = 6000.0

rc_params = dict(
    b=b_ref, h=h_ref,
    A_s_bot=A_s_bot_ref, z_s_bot=z_s_bot,
    A_s_top=A_s_top_ref, z_s_top=z_s_top,
    f_ck=30.0, f_yk=500.0,
    L_a=L_ref, L_b=L_ref,
)

# Uniform doubly-reinforced beam — design / characteristic / mean
triple_uni = ContinuousBeamModes.for_rc(**rc_params)
cba_uni = triple_uni.design

# Variable-CS beam — design / characteristic / mean
triple_var = ContinuousBeamVarCSModes.for_rc(**rc_params)
cba_var = triple_var.design

print('Models built.')

## x/d and ω comparison — hogging section

EC2 singly-reinforced reference:  $x/d = \omega / 0.8$  with
$\omega = \rho \cdot f_{yd} / f_{cd}$
(top bars only, $d = h - z_{s,top}$).

In [ ]:

# Effective depth for hogging: from bottom compression face to top tension bars
# z_s_top is measured from the bottom, so d_hog = z_s_top (NOT h - z_s_top)
d_hog = z_s_top   # = 650 mm

# Material design values from the var CS hogging CS (cs_upper = top bars only)
cs_hog = cba_var.span_left.cs
f_cd = cs_hog.concrete.f_cd                          # alpha_cc * f_ck / gamma_c = 17 MPa
f_yd = cs_hog.reinforcement.layers[0].material.f_yd  # f_yk / gamma_s ≈ 435 MPa

rho_top  = A_s_top_ref / (b_ref * d_hog)
omega_top = rho_top * f_yd / f_cd                    # hogging mechanical ratio (top bars only)
xd_ec2   = omega_top / 0.8                           # EC2 rectangular stress block

print('=== x/d at hogging section — design mode ===')
print(f'  d_hog = {d_hog:.0f} mm,  f_cd = {f_cd:.2f} MPa,  f_yd = {f_yd:.2f} MPa')
print(f'  ρ_top = {rho_top:.5f},  ω_top = {omega_top:.4f}')
print()
print(f'  uniform  (doubly reinforced)  x/d = {cba_uni.xd_hog_u:.4f}')
print(f'  var CS   (top bars only)      x/d = {cba_var.xd_hog_u:.4f}')
print(f'  EC2 ref  (ω_top / 0.8)        x/d = {xd_ec2:.4f}')
print()
print(f'  Δ(var CS vs EC2 ref):   Δ(x/d) = {cba_var.xd_hog_u - xd_ec2:+.4f}')
print(f'  Δ(uniform vs var CS):   Δ(x/d) = {cba_uni.xd_hog_u - cba_var.xd_hog_u:+.4f}  ← compression-bar effect')


## Ultimate load and rotation summary

In [ ]:

q_uni = cba_uni.q_hog_u
q_var = cba_var.q_hog_u

theta_u_uni  = cba_uni.theta_u_at_step(-1)
theta_el_uni = cba_uni.theta_el
theta_pl_uni = cba_uni.theta_pl_at_step(-1)
kappa_hog_uni = float(cba_uni.sweep['kappa_hog'][-1]) * 1e3

theta_u_var  = cba_var.theta_u_at_step(-1)
theta_el_var = cba_var.theta_el
theta_pl_var = cba_var.theta_pl_at_step(-1)
kappa_hog_var = float(cba_var.sweep['kappa_hog'][-1]) * 1e3

# ω based on hogging tension steel (top bars) — consistent EC2 definition
# omega_top already computed in the x/d cell above

def _print_row(label, q, xd, kappa_hog, theta_el, theta_u, theta_pl):
    print(f'  {label}')
    print(f'    q_hog_u   = {q:.4f} kN/m')
    print(f'    x/d       = {xd:.4f}')
    print(f'    κ_R,hog   = {kappa_hog:.4f} ‰/mm')
    print(f'    θ_el      = {theta_el*1e3:.3f} mrad')
    print(f'    θ_u       = {theta_u*1e3:.3f} mrad')
    print(f'    θ_pl      = {theta_pl*1e3:.3f} mrad')

print(f'  (both models: ω_top = {omega_top:.4f},  EC2 x/d ref = {xd_ec2:.4f})')
print()
print('=== ULS state — design mode ===')
_print_row('uniform  (doubly reinforced)', q_uni,
           cba_uni.xd_hog_u, kappa_hog_uni, theta_el_uni, theta_u_uni, theta_pl_uni)
print()
_print_row('var CS   (segmented reinf.)', q_var,
           cba_var.xd_hog_u, kappa_hog_var, theta_el_var, theta_u_var, theta_pl_var)


## Bending moment profiles at ULS

In [ ]:

def _ann_M(ax, color, xd, omega):
    ax.text(0.01, 0.03,
        fr'$x/d$ = {xd:.4f}   $\omega$ = {omega:.4f}',
        transform=ax.transAxes, ha='left', va='bottom', fontsize=9, color=color,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color, alpha=0.90))

fig, (ax_u, ax_v) = plt.subplots(1, 2, figsize=(14, 4))

cba_uni.plot_moment_profile_at_step(ax=ax_u, kappa_idx=-1, color='#1f77b4')
ax_u.set_title(fr'$M(x)$ — uniform  ($q$ = {q_uni:.3f} kN/m)', fontsize=10)
_ann_M(ax_u, '#1f77b4', cba_uni.xd_hog_u, omega_top)

cba_var.plot_moment_profile_at_step(ax=ax_v, kappa_idx=-1, color='#d62728')
ax_v.set_title(fr'$M(x)$ — var CS  ($q$ = {q_var:.3f} kN/m)', fontsize=10)
_ann_M(ax_v, '#d62728', cba_var.xd_hog_u, omega_top)

plt.tight_layout()
plt.show()


## Curvature profiles at ULS

In [ ]:

def _ann_kappa(ax, color, xd, omega, kappa_hog, theta_el, theta_u, theta_pl):
    ax.text(0.01, 0.03,
        fr'$x/d$ = {xd:.4f}   $\omega$ = {omega:.4f}',
        transform=ax.transAxes, ha='left', va='bottom', fontsize=8.5, color=color,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color, alpha=0.90))
    ax.text(0.99, 0.97,
        fr'$\kappa_{{R,hog}}$ = {kappa_hog:.4f} ‰/mm'
        '\n'
        fr'$\theta_{{el}}$  = {theta_el * 1e3:.3f} mrad'
        '\n'
        fr'$\theta_{{u}}$   = {theta_u  * 1e3:.3f} mrad'
        '\n'
        fr'$\theta_{{pl}}$  = {theta_pl * 1e3:.3f} mrad',
        transform=ax.transAxes, ha='right', va='top', fontsize=8.5, color=color,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color, alpha=0.90))

fig, (ax_u, ax_v) = plt.subplots(1, 2, figsize=(14, 4))

cba_uni.plot_kappa_profile_at_step(ax=ax_u, kappa_idx=-1, color='#1f77b4',
                                    show_annotations=False)
ax_u.set_title(fr'$\kappa(x)$ — uniform  ($q$ = {q_uni:.3f} kN/m)', fontsize=10)
_ann_kappa(ax_u, '#1f77b4', cba_uni.xd_hog_u, omega_top,
           kappa_hog_uni, theta_el_uni, theta_u_uni, theta_pl_uni)

cba_var.plot_kappa_profile_at_step(ax=ax_v, kappa_idx=-1, color='#d62728',
                                    show_annotations=False)
ax_v.set_title(fr'$\kappa(x)$ — var CS  ($q$ = {q_var:.3f} kN/m)', fontsize=10)
_ann_kappa(ax_v, '#d62728', cba_var.xd_hog_u, omega_top,
           kappa_hog_var, theta_el_var, theta_u_var, theta_pl_var)

plt.tight_layout()
plt.show()


## Design / characteristic / mean comparison — variable CS beam

Same plot as notebook `01_uls_state.ipynb` but using the segmented-reinforcement model.
The three modes share the same top-bar ω; the differences in κ_R,hog and θ reflect
the different concrete and steel strength models.

In [ ]:
fig, (ax_M, ax_k) = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Variable-CS beam — design / characteristic / mean', fontsize=11)

modes = list(triple_var.iter_modes())
for row, (cba, color, suffix) in enumerate(modes):
    q_mode     = cba.q_hog_u
    kappa_mode = float(cba.sweep['kappa_hog'][-1]) * 1e3
    theta_el_m = cba.theta_el
    theta_pl_m = cba.theta_pl_at_step(-1)
    mode_name  = suffix.strip()[1:-1]    # 'design' / 'characteristic' / 'mean'

    cba.plot_moment_profile_at_step(ax=ax_M, kappa_idx=-1, color=color,
                                    label_suffix=suffix, show_annotations=False)
    cba.plot_kappa_profile_at_step(ax=ax_k, kappa_idx=-1, color=color,
                                   label_suffix=suffix, show_annotations=False)

    ax_k.text(0.99, 0.97 - 0.22 * row,
        fr'$\kappa_{{R,hog}}$ = {kappa_mode:.4f}  '
        fr'$\theta_{{pl}}$ = {theta_pl_m*1e3:.3f} mrad  '
        fr'$q$ = {q_mode:.1f} kN/m  ({mode_name})',
        transform=ax_k.transAxes, ha='right', va='top', fontsize=8, color=color,
        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor=color, alpha=0.85))
    ax_M.text(0.99, 0.97 - 0.10 * row,
        fr'$q$ = {q_mode:.1f} kN/m  ({mode_name})',
        transform=ax_M.transAxes, ha='right', va='top', fontsize=8, color=color,
        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor=color, alpha=0.85))

ax_M.set_title(r'$M(x)$ — var CS', fontsize=10)
ax_k.set_title(r'$\kappa(x)$ — var CS', fontsize=10)
plt.tight_layout()
plt.show()